In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import datetime
from pyquaternion import Quaternion
import plotly.express as px
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import argparse

In [ ]:
report_df = pd.DataFrame(columns=['method', 'stage', 'user', 'score', 'scoreTags',
                                  'manualCnt', 'handCnt', 'IdleCnt', 'TargetCnt',
                                  'manualInCnt', 'handInCnt', 'IdleInCnt', 'TargetInCnt',
                                  'manualOutCnt', 'handOutCnt', 'IdleOutCnt', 'TargetOutCnt'
                                  ])

In [ ]:
config_list = [
  'startedTime',
  'user',
  'stage',
  'beginScore',
  'mode',
  'fov',
  'method',
  'interaxial',
  'sensitivity',
  'frameLength',
]

result={
  'method': 'na',
  'stage': 'na',
  'user': 'na',
  'score': -1,
  'widePortion': -1,
  'transitionPortion': -1,
  'transitionCount': -1,
  'head_qu_distance': -1,
  'head_qu_mean_velocity': -1,
  'head_qu_mean_acc': -1,
  'rightHand_qu_distance': -1,
  'rightHand_qu_mean_velocity': -1,
  'rightHand_qu_mean_acc': -1,
  'leftHand_qu_distance': -1,
  'leftHand_qu_mean_velocity': -1,
  'leftHand_qu_mean_acc': -1,
  'player_mean_velocity': -1,
  'player_mean_acc': -1,
}

In [ ]:
userList = [
  'P008',
  'P017',
  'P036',
  'P034',
  'P005',
  'P019',
  'P015',
  'P031',
  'P018',
  'P025',
  'P006',
  'P030',
  'P014',
  'P010'
]

methodList = [
  'Dynamic'
]

In [ ]:
for userIdx in range(len(userList)):
  for methodIdx in range(len(methodList)):
    arg = "../data/chi24/summary/LOG_Sum_" + userList[userIdx] + "_" + methodList[methodIdx] + ".txt"
    config = {}
    unstruct_tags = []
    unstruct_zoom_tags = []
    duration = 0.0

    # completeFileName = args.file
    completeFileName = arg

    f = open(completeFileName, 'r')
    count = 0
    frameCount = 0
    for i, line in enumerate(f):
      if i < len(config_list) and i == count:
        if (line.find("startedTime") == 0):
          startedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
      if i >= len(config_list) and i == count:
        if (line.find("endedTime") == 0):
          endedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
        if (line.find("*") != 0 and line.find("#") != 0):
          frameCount += 1
      count += 1

    # print("startedTime: ", startedTime)
    # print("endedTime: ", endedTime)
    actualDuration = (endedTime - startedTime).total_seconds()
    # print("frameCount: ", frameCount)
    # print("actual duration: ", actualDuration, "seconds")
    actualFrameRate = frameCount / actualDuration # in FPS
    # print("actual frameRate: ", actualFrameRate, "FPS")
    turncatePoint = round(240 * actualFrameRate)
    # print("trucate point frame: ", turncatePoint)

    f = open(completeFileName, 'r')
    count = 0
    frameCount = 0
    for i, line in enumerate(f):
      if (frameCount > turncatePoint):
        break
      if (line == "===\n"):
        break
      if i < len(config_list) and i == count:
        config[config_list[count]] = line[line.find(":")+1:].strip('\n')
      if i >= len(config_list) and i == count:
        if (line.find("*") == 0):
          unstruct_tags.append(str(frameCount) + line.split("\n")[0])
        elif (line.find("#") == 0):
          unstruct_zoom_tags.append(str(frameCount) + line.split("\n")[0])
        else:
          frameCount += 1
      count += 1

    # check fov
    if (config['mode'] == 'Static'):
      config.update({'method': 'N/A'})
      config.update({'sensitivity': 'N/A'})
      if (config['fov'] != 'Normal'):
        print("\n⚠️ STATIC FOV IS NOT NORMAL")
    elif (config['mode'] == 'Dynamic'):
      config.update({'fov': 'N/A'})

    # check duration
    config.update({'frameLength': (1/actualFrameRate)})
    config.update({'duration': duration}) # trucated duration

    # check score
    if (config['stage'] == '3 maze'):
      config.update({'score': duration})
    else:
      config.update({'score': len(unstruct_tags)})
    if (config['beginScore'] != '0 Pt'):
      print("\n⚠️ BEGIN SCORE IS NOT 0")

    tags = []

    for i, k in enumerate(unstruct_tags):
      tags.append(int(k.split('*')[0]))

    print(tags)

    if (len(tags) != config['score'] and config['stage'] != '3 maze'):
      print("\n⚠️ TAGS LENGTH AND CONFIG SCORE IS NOT EQUAL")
    else:
      print("\n✅ tags length ok")

    zoom_tags = []
    zoom_tags_frame = set()

    for i, k in enumerate(unstruct_zoom_tags):
      zoom_tags.append({"frame": int(k.split('#')[0]), "zoom_tag": k.split('#')[1]})
      zoom_tags_frame.add(int(k.split('#')[0]))

    if (len(zoom_tags) > 0):
      zoom_tags_df = pd.DataFrame(zoom_tags)
      zoom_tags_df = zoom_tags_df.groupby(['frame'], as_index=False).agg(lambda x: '/'.join(x.tolist()))
    
    targetCnt = 0
    idleCnt = 0
    handsCnt = 0
    manualCnt = 0
    targetOutCnt = 0
    targetInCnt = 0
    idleOutCnt = 0
    idleInCnt = 0
    handsOutCnt = 0
    handsInCnt = 0
    manualOutCnt = 0
    manualInCnt = 0

    for i, k in enumerate(zoom_tags):
      if (k['zoom_tag'].find('Ta') != -1):
        targetCnt += 1
        if (k['zoom_tag'].find('Out') != -1):
          targetOutCnt += 1
        else:
          targetInCnt += 1
      elif (k['zoom_tag'].find('Id') != -1):
        idleCnt += 1
        if (k['zoom_tag'].find('Out') != -1):
          idleOutCnt += 1
        else:
          idleInCnt += 1
      elif (k['zoom_tag'].find('Ha') != -1):
        handsCnt += 1
        if (k['zoom_tag'].find('Out') != -1):
          handsOutCnt += 1
        else:
          handsInCnt += 1
      elif (k['zoom_tag'].find('Ma') != -1):
        manualCnt += 1
        if (k['zoom_tag'].find('Out') != -1):
          manualOutCnt += 1
        else:
          manualInCnt += 1

    report_df = pd.concat([report_df, pd.DataFrame([{ 'method': config['method'], 'stage': config['stage'], 'user': config['user'], 'score': config['score'], 'scoreTags': tags, 
                                                     'manualCnt': manualCnt, 'handCnt': handsCnt, 'IdleCnt': idleCnt, 'TargetCnt': targetCnt,
                                                     'manualInCnt': manualInCnt, 'handInCnt': handsInCnt, 'IdleInCnt': idleInCnt, 'TargetInCnt': targetInCnt,
                                                     'manualOutCnt': manualOutCnt, 'handOutCnt': handsOutCnt, 'IdleOutCnt': idleOutCnt, 'TargetOutCnt': targetOutCnt
                                                     }])], ignore_index=True)

In [ ]:
report_df

In [ ]:
print("total triggerd times of manaul in", report_df['manualInCnt'].sum())
print("total triggerd times of hands in", report_df['handInCnt'].sum())
print("total triggerd times of idle in", report_df['IdleInCnt'].sum())
print("total triggerd times of target in", report_df['TargetInCnt'].sum())
print("total triggerd times of manaul out", report_df['manualOutCnt'].sum())
print("total triggerd times of hands out", report_df['handOutCnt'].sum())
print("total triggerd times of idle out", report_df['IdleOutCnt'].sum())
print("total triggerd times of target out", report_df['TargetOutCnt'].sum())

In [ ]:
fig, ax = plt.subplots()

methods = ['Manual', 'Target']
counts = [report_df['manualOutCnt'].sum(), report_df['TargetOutCnt'].sum()]
bar_labels = ['Manual', 'Target']
# bar_colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:orange']

ax.bar(methods, counts, label=bar_labels, color='#256A96')

ax.set_ylabel('Triggered times')
ax.set_title('Zoom out')
# ax.legend(title='Methods')

# change fig size
fig = plt.gcf()
fig.set_size_inches(4, 5)

plt.show()

In [ ]:
fig, ax = plt.subplots()

methods = ['Arms', 'Moving']
counts = [report_df['handInCnt'].sum(), report_df['IdleInCnt'].sum()]
bar_labels = ['Arms', 'Moving']
# bar_colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:orange']

ax.bar(methods, counts, label=bar_labels, color='#256A96')

ax.set_ylabel('Triggered times')
ax.set_title('Zoom in')
# ax.legend(title='Methods')

# change the y axis range
ax.set_ylim([0, 2000])

# change fig size
fig = plt.gcf()
fig.set_size_inches(4, 5)

plt.show()